## Использование Gamac на реальных данных
Описание реальных задач приведено в [документации](https://github.com/ITMO-CODE-AI/GaMAC/blob/develop/docs/CASE_RU.md)

### Импортируем нужные библиотеки

In [ ]:
import os
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
from PIL import Image

from gamac.autoclustering import Gamac
from gamac.estimation.internal import Internal

### 1. Данные промышленных логов (семпл)
Кластеризация логов направлена на автоматическое выделение групп записей, связанных с общими источниками событий, такими как прикладные процессы, системные события или среды исполнения. Основная цель — обнаружить скрытые паттерны в данных, которые позволяют косвенно определить природу возникновения логов, даже если их формат не содержит явных указаний на источник.

Для получения данных нужно выгрузить их из Minio, данные лежат в: datasets/data/synthetic_logs_1kk.csv

И переложить из корня в папку data/

Примечание: выполнение по этому датасету проходит очень долгое время в связи с его объемом

In [ ]:
# Импортируем датафрейм
data = pd.read_csv('../data/synthetic_logs_1kk.csv')

In [ ]:
data.head(5)

In [ ]:
# По умолчанию кол-во итераций стоит 50
gamac = Gamac(iter_limit=100, target_measures={Internal.BR})

result = gamac.run(table=data)

In [ ]:
# Получение топ-50 меток по лучшему классификатору
print(result.labels[:50])

### 2. Данные по анализу цветов RGB-изображений
Кластеризация цветов RGB-изображений направлена на автоматическое группирование пикселей со схожими цветовыми значениями в отдельные кластеры, где каждый кластер представляет собой усреднённый цвет или доминирующий оттенок из соответствующей группы. Основная цель — упростить представление изображения за счёт выделения ключевых цветовых паттернов, что может быть полезно для задач сжатия данных, сегментации объектов, снижения шума или анализа цветовой палитры.

In [ ]:
# Импортируем картинки
images = []

# Получаем список файлов в директории
for filename in os.listdir('../data/images/'):
    if 'png' not in filename:
        continue
    file_path = os.path.join('../data/images/', filename)

    # Пытаемся открыть файл как изображение
    with Image.open(file_path) as img:
        images.append(img.copy())

images[:5]

### Запустим подбор

In [ ]:
# Для кластеризации передадим константный пустой текст
gamac = Gamac(iter_limit=30)
empty_texts = ['' for i in range(len(images))]

result = gamac.run(image=images, text=empty_texts)

In [ ]:
# Метки по датасету
print(result.labels)